# Data Augmentation Pipeline

1. **Stage 1** – Spatial translation (`DataAugmentation.m`)
2. **Stage 2** – Distribution-based generation (`DataDistribution.m`)
3. **Stage 3** – Spatial translation on distributed data (`DataDistributionAugmentation.m`)

Each stage can be run independently so that intermediate outputs can be inspected.

The final dataset is guaranteed to have no negative pressure values.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
 
from bedposip.data.stages import (
    spatial_augmentation,
    distribution_generation,
    dist_spatial_augmentation,
)
from bedposip.data.balancer import balance_to_target

I0000 00:00:1785128073.489525   87177 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Configuration

In [2]:
# Adjustable parameters
INPUT_CSV = "input/DatosEntrenamiento.csv"            # original 192-row dataset
OUTPUT_DIR = Path("input/")                           # where final CSV is saved
RANDOM_STATE = 42

# Stage 2 parameters
N_SAMPLES = 6000       # samples per centroid category (default = 1000 → ~25 k total)
CAT_THR = 10           # minimum observations per centroid category

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Original Data

In [3]:
df_input = pd.read_csv(INPUT_CSV)
print(f"Original shape: {df_input.shape}")
print("Class distribution:")
print(df_input["POSICION"].value_counts())

Original shape: (192, 65)
Class distribution:
POSICION
trunk_L    34
trunk_R    32
fetus_R    32
prone      32
fetus_L    31
supine     31
Name: count, dtype: int64


## Stage 1 – Spatial Translation

In [4]:
rng = np.random.default_rng(RANDOM_STATE)

df_stage1 = spatial_augmentation(df_input, rng=rng)

print(f"Stage 1 shape: {df_stage1.shape}")
print("Class distribution:")
print(df_stage1["POSICION"].value_counts())

Stage 1 shape: (626, 65)
Class distribution:
POSICION
fetus_R    156
trunk_L    122
trunk_R     97
fetus_L     97
supine      80
prone       74
Name: count, dtype: int64


## Stage 2 – Distribution-Based Generation

In [5]:
df_stage2 = distribution_generation(
    df_stage1,
    n_samples=N_SAMPLES,
    cat_thr=CAT_THR,
    rng=rng,
)

print(f"Stage 2 shape: {df_stage2.shape}")
print("Class distribution:")
print(df_stage2["POSICION"].value_counts())

Stage 2 shape: (150000, 65)
Class distribution:
POSICION
trunk_L    36000
trunk_R    24000
fetus_L    24000
fetus_R    24000
supine     24000
prone      18000
Name: count, dtype: int64


## Stage 3 – Spatial Translation on Distributed Data

In [6]:
df_final = dist_spatial_augmentation(df_stage2, rng=rng)

print(f"Final shape: {df_final.shape}")
print("Class distribution:")
print(df_final["POSICION"].value_counts())

Final shape: (640094, 65)
Class distribution:
POSICION
fetus_L    130273
trunk_L    130239
supine     113743
trunk_R    112135
fetus_R    102483
prone       51221
Name: count, dtype: int64


## Validation

In [7]:
feature_cols = [c for c in df_final.columns if c != "POSICION"]

# 1. No negative values
min_val = df_final[feature_cols].min().min()
assert min_val >= 0.0, f"Negative value found: {min_val}"
print(f"✓ Min value: {min_val:.4f} (no negatives)")

# 2. Expected column count (64 features + POSICION)
assert len(df_final.columns) == 65
print("✓ 65 columns (64 features + POSICION)")

# 3. All 6 classes present
expected_classes = {"trunk_L", "trunk_R", "fetus_L", "fetus_R", "supine", "prone"}
actual_classes = set(df_final["POSICION"].unique())
assert actual_classes == expected_classes
print(f"✓ All 6 classes present: {actual_classes}")

✓ Min value: 0.0000 (no negatives)
✓ 65 columns (64 features + POSICION)
✓ All 6 classes present: {'fetus_R', 'trunk_R', 'fetus_L', 'prone', 'trunk_L', 'supine'}


## Save Final Dataset

In [8]:
output_path = OUTPUT_DIR / "DatosEntrenamiento_DistProbTransEspacial.csv"
df_final.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: input/DatosEntrenamiento_DistProbTransEspacial.csv


## Balance to Target

In [9]:
# If any class has fewer than the target, a ValueError is raised
# telling you how many samples are short.

CLASS_SAMPLES = 50000

TARGETS = {
    "trunk_L": CLASS_SAMPLES,
    "trunk_R": CLASS_SAMPLES,
    "fetus_L": CLASS_SAMPLES,
    "fetus_R": CLASS_SAMPLES,
    "supine": CLASS_SAMPLES,
    "prone": CLASS_SAMPLES,
}

balanced = balance_to_target(df_final, TARGETS, random_state=RANDOM_STATE)
print(f"Balanced shape: {balanced.shape}")
print(balanced["POSICION"].value_counts())

Balanced shape: (300000, 65)
POSICION
trunk_L    50000
trunk_R    50000
fetus_L    50000
fetus_R    50000
supine     50000
prone      50000
Name: count, dtype: int64


In [10]:
output_path = OUTPUT_DIR / "DatosEntrenamiento_DistProbTransEspacial_balanced.csv"
balanced.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: input/DatosEntrenamiento_DistProbTransEspacial_balanced.csv
